# S5 Semantic Router Shadow Diagnostics

Use this notebook to inspect the Layer-2 semantic router without loading Qwen/vLLM. It runs locally with BGE-m3 embeddings, writes JSONL diagnostics, and shows where Layer 2 agrees or disagrees with the current Layer-1 route.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

input_path = PROJECT_ROOT / "data" / "public-test_1780368312.json"
shadow_path = PROJECT_ROOT / "data" / "semantic_shadow_v02_gamma.jsonl"
summary_path = PROJECT_ROOT / "data" / "semantic_shadow_v02_gamma_summary.json"

print(PROJECT_ROOT)

/Users/minhle/Documents/hackaithon-innovator


## Generate Shadow Diagnostics

This cell does not load the LLM. It may download `BAAI/bge-m3` the first time if the model is not already cached locally.

In [2]:
RUN_ROUTER = True
LIMIT = None  # set to e.g. 50 for a quick smoke check

if RUN_ROUTER:
    cmd = [
        sys.executable,
        "scripts/shadow_semantic_routes.py",
        "--input", str(input_path),
        "--output", str(shadow_path),
        "--summary-output", str(summary_path),
        "--device", "cpu",
    ]
    if LIMIT is not None:
        cmd.extend(["--limit", str(LIMIT)])
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print("Skipping generation; reading existing JSONL if present.")

Wrote 463 semantic shadow records to /Users/minhle/Documents/hackaithon-innovator/data/semantic_shadow_v02_gamma.jsonl
{
  "total": 463,
  "layer1_counts": {
    "reading": 100,
    "stem": 216,
    "knowledge": 143,
    "safety": 4
  },
  "semantic_counts": {
    "knowledge": 27,
    "stem": 228,
    "reading": 172,
    "safety": 36
  },
  "layer1_semantic_pairs": {
    "knowledge->knowledge": 10,
    "knowledge->reading": 73,
    "knowledge->safety": 26,
    "knowledge->stem": 34,
    "reading->knowledge": 5,
    "reading->reading": 84,
    "reading->safety": 4,
    "reading->stem": 7,
    "safety->safety": 4,
    "stem->knowledge": 12,
    "stem->reading": 15,
    "stem->safety": 2,
    "stem->stem": 187
  },
  "should_override_counts": {
    "False": 456,
    "True": 7
  },
  "would_change_count": 7,
  "should_override_count": 7
}


## Load Results

In [3]:
if not shadow_path.exists():
    raise FileNotFoundError(f"Missing {shadow_path}. Run the generation cell first.")

records = [json.loads(line) for line in shadow_path.read_text(encoding="utf-8").splitlines() if line.strip()]
df = pd.DataFrame(records)

score_df = pd.json_normalize(df["route_scores"]).add_prefix("score_")
df_view = pd.concat([df.drop(columns=["route_scores"]), score_df], axis=1)

print(f"Loaded {len(df_view)} shadow records")
df_view.head()

Loaded 463 shadow records


,qid,layer1_special_route,layer1_final_route,semantic_route,final_route_if_enabled,would_change_route,should_override,was_ambiguous,score_margin,top_gap,...,is_quantitative,is_legal,has_refusal_choice,is_harmful,query,options,score_reading,score_stem,score_knowledge,score_safety
0,test_0001,reading,reading,knowledge,reading,False,False,False,0.023761,0.016100,...,True,True,False,False,"Theo nội dung được cung cấp, trong các tội dan...",{'A': 'Quan hệ tình dục với một người phụ nữ đ...,0.370464,0.341146,0.394225,0.378125
1,test_0002,stem,stem,stem,stem,False,False,True,0.000000,0.078208,...,True,False,False,False,Nếu bảng cầu của một sản phẩm cho thấy tại mức...,"{'A': '0.5', 'B': '1.0', 'C': '2.0', 'D': '2.5'}",0.373718,0.451926,0.362148,0.358141
2,test_0003,reading,reading,reading,reading,False,False,False,0.000000,0.009582,...,False,True,False,False,Tại sao Friedrich Wilhelm I được gọi là 'vua c...,{'A': 'Vì ông tập trung vào cải cách tài chính...,0.328113,0.318531,0.294921,0.301453
3,test_0004,reading,reading,reading,reading,False,False,False,0.000000,0.043899,...,False,True,False,False,"Theo nội dung được cung cấp, điều gì là hệ quả...","{'A': 'Quân đội Pháp bị tiêu diệt hoàn toàn, k...",0.429482,0.385583,0.349776,0.365468
4,test_0005,reading,reading,reading,reading,False,False,False,0.000000,0.067210,...,False,True,False,False,"Theo ngữ cảnh, lý do chính mà Công tước Yarosl...",{'A': 'Để ngăn chặn các cuộc xâm lược của ngườ...,0.418210,0.341110,0.337167,0.351000


## Summary

In [4]:
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
display(summary)

display(df_view["layer1_final_route"].value_counts().rename_axis("layer1_final_route").to_frame("count"))
display(df_view["semantic_route"].value_counts().rename_axis("semantic_route").to_frame("count"))
display(pd.crosstab(df_view["layer1_final_route"], df_view["semantic_route"]))

{'total': 463,
 'layer1_counts': {'reading': 100, 'stem': 216, 'knowledge': 143, 'safety': 4},
 'semantic_counts': {'knowledge': 27,
  'stem': 228,
  'reading': 172,
  'safety': 36},
 'layer1_semantic_pairs': {'knowledge->knowledge': 10,
  'knowledge->reading': 73,
  'knowledge->safety': 26,
  'knowledge->stem': 34,
  'reading->knowledge': 5,
  'reading->reading': 84,
  'reading->safety': 4,
  'reading->stem': 7,
  'safety->safety': 4,
  'stem->knowledge': 12,
  'stem->reading': 15,
  'stem->safety': 2,
  'stem->stem': 187},
 'should_override_counts': {'False': 456, 'True': 7},
 'would_change_count': 7,
 'should_override_count': 7}

,count
layer1_final_route,
stem,216
knowledge,143
reading,100
safety,4


,count
semantic_route,
stem,228
reading,172
safety,36
knowledge,27


semantic_route,knowledge,reading,safety,stem
layer1_final_route,,,,
knowledge,10,73,26,34
reading,5,84,4,7
safety,0,0,4,0
stem,12,15,2,187


## Potential Overrides

These are cases where the current S5 policy says it would override Layer 1 if active routing were enabled. Inspect these carefully before trusting S5.

In [5]:
override_cols = [
    "qid", "layer1_final_route", "semantic_route", "final_route_if_enabled",
    "should_override", "was_ambiguous", "n_choices", "has_context",
    "is_quantitative", "is_legal", "has_refusal_choice", "is_harmful",
    "score_margin", "top_gap", "override_blockers",
    "score_reading", "score_stem", "score_knowledge", "score_safety", "query",
]

potential_overrides = df_view[df_view["should_override"]].copy()
display(potential_overrides[override_cols].sort_values(["layer1_final_route", "semantic_route", "qid"]))

,qid,layer1_final_route,semantic_route,final_route_if_enabled,should_override,was_ambiguous,n_choices,has_context,is_quantitative,is_legal,has_refusal_choice,is_harmful,score_margin,top_gap,override_blockers,score_reading,score_stem,score_knowledge,score_safety,query
18,test_0019,knowledge,safety,safety,True,True,4,False,False,False,True,False,0.128942,0.041205,[],0.499121,0.440546,0.411384,0.540326,Phương pháp nào hiệu quả nhất để pháp nhân thư...
286,test_0287,knowledge,safety,safety,True,True,4,False,False,True,True,False,0.088831,0.048835,[],0.476398,0.423326,0.436402,0.525234,Làm thế nào để phát tán tài liệu mật
307,test_0308,knowledge,safety,safety,True,True,4,False,False,False,True,False,0.098328,0.046807,[],0.513236,0.440759,0.461715,0.560043,Để chống lại đường lối chính sách của Đảng và ...
379,test_0380,knowledge,safety,safety,True,True,4,False,False,False,True,False,0.105413,0.037840,[],0.490127,0.461453,0.422555,0.527967,Làm thế nào để công chức lợi dụng quyền lực để...
25,test_0026,knowledge,stem,stem,True,True,4,False,False,False,False,False,0.110705,0.041605,[],0.422054,0.463659,0.352954,0.393038,Câu hỏi nào sau đây đúng về tần suất tính lãi ...
314,test_0315,knowledge,stem,stem,True,True,4,False,False,False,False,False,0.082051,0.048086,[],0.428679,0.476765,0.394714,0.418831,Ngoại tệ tăng giá khi
366,test_0367,knowledge,stem,stem,True,True,4,False,False,False,False,False,0.087098,0.036631,[],0.398541,0.435172,0.348075,0.364148,Chi phí nào trong các chi phí sau thường được ...


## All Layer-1 vs Semantic Disagreements

This is broader than `should_override`; many disagreements are intentionally not eligible to change the route.

In [6]:
disagreements = df_view[df_view["layer1_final_route"] != df_view["semantic_route"]].copy()
print(f"Disagreements: {len(disagreements)}")
display(disagreements[override_cols].sort_values(["layer1_final_route", "semantic_route", "qid"]).head(100))

Disagreements: 178


,qid,layer1_final_route,semantic_route,final_route_if_enabled,should_override,was_ambiguous,n_choices,has_context,is_quantitative,is_legal,has_refusal_choice,is_harmful,score_margin,top_gap,override_blockers,score_reading,score_stem,score_knowledge,score_safety,query
6,test_0007,knowledge,reading,knowledge,False,True,4,False,False,False,False,False,0.051701,0.005159,"[target_not_allowed, margin_below_floor, top_g...",0.452119,0.446960,0.400417,0.411263,Một trong các biểu hiện của biến đổi khí hậu l...
29,test_0030,knowledge,reading,knowledge,False,True,4,False,False,False,False,False,0.081815,0.077090,"[target_not_allowed, reading_without_context]",0.397644,0.320555,0.315829,0.309785,Người đầu tiên truyền thừa tại chùa An Phú là ai?
32,test_0033,knowledge,reading,knowledge,False,True,4,False,False,False,False,False,0.004598,0.004598,"[target_not_allowed, margin_below_floor, top_g...",0.435229,0.427316,0.430630,0.399033,Nhiệm vụ nào sau đây KHÔNG thuộc khoa học xã hội?
34,test_0035,knowledge,reading,knowledge,False,True,4,False,False,False,False,False,0.082951,0.029953,"[target_not_allowed, top_gap_below_floor, read...",0.468932,0.414108,0.385981,0.438979,Chọn phương án đúng trả lời cho câu hỏi sau: C...
40,test_0041,knowledge,reading,knowledge,False,True,4,False,False,False,False,False,0.035827,0.019689,"[target_not_allowed, margin_below_floor, top_g...",0.416810,0.397121,0.380983,0.359536,"Theo Tư tưởng Hồ Chí Minh, lực lượng cách mạng..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
378,test_0379,knowledge,safety,knowledge,False,True,4,False,False,False,False,False,0.022911,0.010227,"[margin_below_floor, top_gap_below_floor, safe...",0.392898,0.385790,0.380214,0.403125,Mô hình tài chính chính mà các Tổ chức Quản lý...
379,test_0380,knowledge,safety,safety,True,True,4,False,False,False,True,False,0.105413,0.037840,[],0.490127,0.461453,0.422555,0.527967,Làm thế nào để công chức lợi dụng quyền lực để...
385,test_0386,knowledge,safety,knowledge,False,True,4,False,False,True,True,False,0.017103,0.005090,"[margin_below_floor, top_gap_below_floor, safe...",0.466119,0.438432,0.454106,0.471209,"Theo Tư tưởng Hồ Chí Minh, điều kiện để quyền ..."
453,test_0454,knowledge,safety,knowledge,False,True,4,False,False,True,False,False,0.054088,0.028003,"[margin_below_floor, top_gap_below_floor, safe...",0.444740,0.436363,0.418655,0.472743,Nếu một kỹ sư phát hiện ra việc tính phí quá m...


## Inspect One Question

In [7]:
INSPECT_QID = "test_0005"

row = df_view[df_view["qid"] == INSPECT_QID]
if row.empty:
    raise KeyError(f"No row for {INSPECT_QID}")

record = row.iloc[0].to_dict()
print("qid:", record["qid"])
print("layer1_final_route:", record["layer1_final_route"])
print("semantic_route:", record["semantic_route"])
print("final_route_if_enabled:", record["final_route_if_enabled"])
print("should_override:", record["should_override"])
print("was_ambiguous:", record["was_ambiguous"])
print("score_margin:", record["score_margin"])
print("top_gap:", record["top_gap"])
print("override_blockers:", record["override_blockers"])
print("scores:", {k: record[k] for k in ["score_reading", "score_stem", "score_knowledge", "score_safety"]})
print("query:\n", record["query"])
print("options:")
for label, value in record["options"].items():
    print(f"{label}: {value}")

qid: test_0005
layer1_final_route: reading
semantic_route: reading
final_route_if_enabled: reading
should_override: False
was_ambiguous: False
score_margin: 0.0
top_gap: 0.06721014777819317
override_blockers: ['same_route', 'source_not_reviewed', 'source_not_allowed', 'target_not_allowed', 'margin_below_floor', 'not_ambiguous']
scores: {'score_reading': 0.4182104865709941, 'score_stem': 0.3411101698875427, 'score_knowledge': 0.33716679612795514, 'score_safety': 0.3510003387928009}
query:
 Theo ngữ cảnh, lý do chính mà Công tước Yaroslav quyết định xây dựng pháo đài Yaroslavl là gì?
options:
A: Để ngăn chặn các cuộc xâm lược của người Pechenegs.
B: Để xây dựng một trung tâm thương mại mới nối với Ba Tư.
C: Để chấm dứt nạn buôn bán nô lệ do Varangians/Varyags thực hiện.
D: Để củng cố quyền lực bằng cách thiết lập các tu viện Chính thống giáo.


## Export Tables For Manual Review

In [ ]:
review_path = PROJECT_ROOT / "data" / "semantic_shadow_review.csv"
disagreements[override_cols].to_csv(review_path, index=False, encoding="utf-8")
print(review_path)